In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.resource_utils import split_datasets_path
from biked_commons.conditioning import conditioning
from biked_commons.design_evaluation.scoring import *

In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)

data_tens = torch.tensor(data.values, dtype=torch.float32, device=device)

In [3]:
evaluator, requirement_names, requirement_types = construct_tensor_evaluator(get_standard_evaluations(device), data.columns)

In [4]:
num_data = data.shape[0]
rider_condition = conditioning.sample_riders(num_data, split="test")
use_case_condition = conditioning.sample_use_case(num_data, split="test")
text_condition = conditioning.sample_text(num_data, split="test")
image_embeddings = conditioning.sample_image_embedding(num_data, split="test")
condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Embedding": image_embeddings}
# condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Text": text_condition}

In [5]:
#calcualte gradient of scores wrt data_tens

data_tens.requires_grad = True
eval_scores = evaluator(data_tens, condition)
score_sum = eval_scores.sum()
score_sum.backward()


In [6]:
#check for infs and nans in data_tens
if torch.any(torch.isnan(data_tens)) or torch.any(torch.isinf(data_tens)):
    print("Data tensor contains NaN or Inf values.")

In [7]:
#get nan indices of data_tens.grad
nan_indices = torch.isnan(data_tens.grad).nonzero(as_tuple=True)
print("nan indices: ", nan_indices)

nan indices:  (tensor([], device='cuda:0', dtype=torch.int64), tensor([], device='cuda:0', dtype=torch.int64))


In [8]:
eval_scores

tensor([[ 2.4160e+01,  3.0486e+00,  7.2878e+01,  ..., -1.0500e+02,
         -5.1000e+02, -5.0000e-01],
        [ 1.0225e+01,  1.0434e+02,  2.6839e+01,  ..., -1.4750e+02,
         -1.5300e+02, -5.0000e-01],
        [ 2.3527e+01,  6.9383e+00,  6.8831e+01,  ..., -1.3750e+02,
         -4.0800e+02, -5.0000e-01],
        ...,
        [ 2.1400e+01,  1.3696e+01,  8.4117e+01,  ..., -8.2000e+01,
         -4.0800e+02, -5.0000e-01],
        [ 2.3330e+01,  3.5404e+01,  8.4238e+01,  ..., -8.2000e+01,
         -4.0800e+02, -5.0000e-01],
        [ 2.2753e+01,  2.7230e+01,  7.7232e+01,  ..., -1.0750e+02,
         -6.1200e+02, -4.9033e-01]], device='cuda:0', grad_fn=<CopySlices>)

In [9]:
isobjective = torch.tensor(requirement_types) == 1
isobjective = isobjective.to(device)
objective_scores = eval_scores[:, isobjective].detach().cpu().numpy()
# constraint_scores = eval_scores[:, ~isobjective].detach().numpy()

In [10]:
main_scorer = construct_scorer(MainScores, get_standard_evaluations(device), data.columns)
detailed_scorer = construct_scorer(DetailedScores, get_standard_evaluations(device), data.columns)

In [11]:
main_scorer(data_tens.detach(), condition)

Hypervolume                     0.338462
Constraint Satisfaction Rate    0.862882
Maximum Mean Discrepancy        0.000000
dtype: float64

In [12]:
detailed_scorer(data_tens.detach(), condition)

Min Objective Score: Drag Force                                                                              9.410693
Min Objective Score: Knee Angle Error                                                                        0.000000
Min Objective Score: Hip Angle Error                                                                        21.361307
Min Objective Score: Arm Angle Error                                                                         0.000000
Min Objective Score: Cosine Similarity to Embedding                                                          0.000031
Min Objective Score: Mass                                                                                    2.777178
Min Objective Score: Planar Compliance                                                                       0.000000
Min Objective Score: Transverse Compliance                                                                   0.000000
Min Objective Score: Eccentric Compliance               